In [1]:
import os
import pyarrow as pa
import pyarrow.parquet as pq
import wrds

WRDS_USERNAME = 'cpehlivanian'      # password is read from ~/.pgpass
START, END = '2024-01-02', '2024-12-31'
source = 'CRSP dsf_v2 (WRDS)'
dirname = 'price_volume'          # a Parquet "store" is a directory of files

In [2]:
def data(sym_root):
    """Daily unadjusted close price and volume for one ticker from crsp.dsf_v2."""
    conn = wrds.Connection(wrds_username=WRDS_USERNAME)
    try:
        df = conn.raw_sql(
            """
            SELECT permno, dlycaldt, dlyprc, dlyvol
            FROM crsp.dsf_v2
            WHERE ticker = %(ticker)s
              AND dlycaldt BETWEEN %(start)s AND %(end)s
            ORDER BY dlycaldt
            """,
            params={'ticker': sym_root, 'start': START, 'end': END},
            date_cols=['dlycaldt'],
        )
    finally:
        conn.close()
    if df.empty:
        raise ValueError(f"No CRSP data for {sym_root} between {START} and {END}")
    if df['permno'].nunique() > 1:     # tickers can be reused by different securities
        raise ValueError(f"{sym_root} maps to several PERMNOs: {df['permno'].unique()}")
    date = df['dlycaldt'].dt.date.to_numpy()
    price = df['dlyprc'].to_numpy(dtype='float64')
    volume = df['dlyvol'].to_numpy(dtype='float64')
    return date, price, volume

In [3]:
#####################################################################
# We will create and inspect a Parquet store.                       #
# HDF5 groups become directories (hive-style: symbol=WMT), each     #
# group's datasets become columns of one Parquet file, and HDF5     #
# attributes become Arrow field / schema metadata.                  #
# Parquet files are immutable: re-running w() overwrites the file   #
# rather than appending to it.                                      #
#####################################################################

# Write
def w(sym_root):

    path = os.path.join(dirname, 'equities', f"symbol={sym_root}")
    os.makedirs(path, exist_ok=True)

    date, price, volume = data(sym_root)

    schema = pa.schema([
        pa.field('date',   pa.date32()),
        pa.field('price',  pa.float64(), metadata={'units': 'USD (unadjusted)',    'source': source}),
        pa.field('volume', pa.float64(), metadata={'units': 'shares (unadjusted)', 'source': source}),
    ], metadata={'start': START, 'end': END})

    table = pa.table({'date': date, 'price': price, 'volume': volume}, schema=schema)

    pq.write_table(table,
                   os.path.join(path, 'data.parquet'),
                   row_group_size=100_000,   # Parquet analogue of HDF5 chunks
                   compression='gzip',
                   compression_level=4)

In [4]:
# Inspect
def i(sym_root):
    for root, _, files in sorted(os.walk(dirname)):
        for f in files:
            print(os.path.relpath(os.path.join(root, f), dirname))
    pf = pq.ParquetFile(os.path.join(dirname, 'equities', f"symbol={sym_root}", 'data.parquet'))
    print(pf.schema_arrow.field('price').metadata[b'units'].decode())
    print(pf.metadata)                 # rows, row groups, sizes -- no data read

In [5]:
# Peek (only reads in the first row group, price column only)
def p(sym_root, ind):
    pf = pq.ParquetFile(os.path.join(dirname, 'equities', f"symbol={sym_root}", 'data.parquet'))
    batch = next(pf.iter_batches(batch_size=ind, columns=['price']))
    first_few = batch.column('price').to_numpy()
    return first_few

In [6]:
# Full read
# Returns numpy ndarrays
def r(sym_root):
    t = pq.read_table(os.path.join(dirname, 'equities'),
                      columns=['price', 'volume'],
                      filters=[('symbol', '==', sym_root)])   # partition pruning
    p = t.column('price').to_numpy()
    v = t.column('volume').to_numpy()
    return p,v

In [7]:
if __name__ == "__main__":
    # Examples

    w("WMT")
    w("MCD")

    i("WMT")
    print(p("WMT", 5))

    p,v = r("WMT")
    p,v = r("MCD")

Loading library list...
Done
Loading library list...
Done
equities/symbol=MCD/data.parquet
equities/symbol=WMT/data.parquet
USD (unadjusted)
  created_by: parquet-cpp-arrow version 17.0.0
  num_columns: 3
  num_rows: 252
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 1221
[159.29 159.3  157.76 156.71 158.25]


In [8]:
p

array([297.04, 294.39, 291.74, 288.99, 291.9 , 290.87, 294.09, 294.15,
       293.47, 291.09, 291.16, 294.36, 300.53, 298.41, 300.05, 300.44,
       297.21, 292.26, 292.31, 294.65, 292.72, 298.08, 297.05, 285.97,
       284.65, 287.33, 291.93, 289.47, 289.44, 287.11, 288.17, 291.86,
       292.02, 292.65, 293.91, 295.92, 297.75, 297.12, 293.76, 295.19,
       292.28, 290.73, 290.94, 292.54, 292.93, 291.11, 292.55, 294.82,
       294.31, 282.86, 281.73, 279.14, 278.58, 282.87, 284.45, 283.53,
       282.63, 278.62, 278.62, 282.02, 281.95, 280.22, 277.74, 275.54,
       270.09, 266.69, 267.56, 269.44, 268.67, 268.62, 267.39, 266.23,
       265.43, 269.95, 270.98, 271.99, 275.58, 276.88, 276.75, 275.6 ,
       273.09, 273.55, 273.04, 274.43, 273.28, 270.32, 269.3 , 267.5 ,
       268.49, 267.95, 275.  , 271.32, 270.66, 273.87, 273.51, 272.38,
       267.87, 265.87, 265.77, 257.93, 258.11, 253.54, 249.37, 252.07,
       258.89, 259.75, 262.72, 259.99, 260.72, 256.21, 253.81, 254.28,
      